# Generate and Evaluate Synthetic Data

In [176]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Generate Synthetic Data

In [177]:
import pandas as pd
from torch.utils.data import DataLoader
from pathlib import Path
import evaluate
from openai import OpenAI
from datasets import Dataset, DatasetDict
from transformers import AdamW
from tqdm.auto import tqdm
from transformers import get_scheduler
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from tools.utils import generate_response
from tools.prompt_templates import generate_negative_prompts_few_shot, generate_positive_prompts_few_shot
import pandas as pd
import re

In [178]:

processed_data_dir = Path('processed_data/multigenre')
metadata = pd.read_csv(processed_data_dir / 'metadata.tsv',sep='\t')
metadata.shape

(226872, 6)

In [179]:
metadata.head(3)

,file_name,platform,genre,tag,sentence_original,sentence_modified
0,DuolingoPrivacyPolicy.docx.txt.clean_education,DuolingoPrivacyPolicy.docx.txt.clean,education,PP,"Privacy Policy General At Duolingo, we care ab...","privacy policy general at duolingo , we care a..."
1,DuolingoPrivacyPolicy.docx.txt.clean_education,DuolingoPrivacyPolicy.docx.txt.clean,education,PP,This Privacy Policy ( Privacy Policy ) details...,this privacy policy ( privacy policy ) details...
2,DuolingoPrivacyPolicy.docx.txt.clean_education,DuolingoPrivacyPolicy.docx.txt.clean,education,PP,"Duolingo, Inc., a company registered at 5900 P...","[mask] [mask] [mask] , a company registered at..."


In [212]:
clause_type = 'class waiver' #'arbitration' | 'opt-out' | 'class waiver' 
annotator = '_TS'
# annotations_df = pd.read_excel(f'annotations/manual/{clause_type}_annotations_gpt4{annotator}.xlsx',
#                                sheet_name=f'{clause_type}_annotations_gpt4', index_col=0)

annotations_df = pd.read_csv(f'annotations/manual/{clause_type}_annotations_gpt4{annotator}.csv', index_col=0)

annotations_df = annotations_df[~annotations_df.labels.isnull()]
annotations_df.labels = annotations_df.labels.astype(int)
annotations_df.replace({'labels': {2: 1}}, inplace=True)
annotations_df.labels.value_counts(normalize=True)

labels
1    0.64
0    0.36
Name: proportion, dtype: float64

In [213]:
annotations_df = annotations_df.merge(metadata, left_on='text', right_on='sentence_modified', how='left')
annotations_df = annotations_df[['sentence_original','labels']]
annotations_df.columns = ['text','labels']

In [214]:
annotations_df.drop_duplicates(subset='text', inplace=True)


In [215]:

annotations_df['text'] = annotations_df['text'].apply(lambda x: re.sub(r'\n', ' ', x))

In [216]:
# def replace_mask_with_company(text):
#     return re.sub(r'(\[mask\]\s)+', r'[company] ', text)

# annotations_df['text'] = annotations_df.text.apply(replace_mask_with_company)

In [217]:

train,test = train_test_split(annotations_df, test_size=0.75, random_state=42)
print('train size',train.labels.value_counts())
print('test size',test.labels.value_counts())

train size labels
1    17
0     9
Name: count, dtype: int64
test size labels
1    52
0    29
Name: count, dtype: int64


## Generate Synthetic Data
### First iteration

In [218]:
definitions = {
    'class waiver' : """A class action waiver is a provision found in some contracts which prohibits a party from filing a class action legal proceeding against the other party, or both parties waiving the right to file class actions against each other. Most class action waiver clauses include this wording or a variation of it: You and we agree that any dispute filed against each other must be on an individual basis and not as a class or collective action.""",
    'opt-out' : """a clause that permits signatories to a contract to opt out of particular provisions, or to terminate the contract early""",
    'arbitration': """In contract law, an arbitration clause is a clause in a contract that requires the parties to resolve their disputes through an arbitration process. Although such a clause may or may not specify that arbitration occur within a specific jurisdiction, it always binds the parties to a type of resolution outside the courts, and is therefore considered a kind of forum selection clause.""",
    'modification': """This clause gives the platform rights to unilaterally change the contract at any time and states how an agreement can be changed or modified."""
}
num_examples = 100

In [186]:
from tools.prompt_templates import *

In [152]:

positive_prompts_zero_shot = generate_positive_prompts_zero_shot(clause_type, definitions, num_examples)
negative_prompts_zeo_shot = generate_negative_prompts_zero_shot(clause_type, definitions, num_examples)
print(negative_prompts_zeo_shot[0])

You are a helpful AI that creates synthetic data by generating legal clauses for terms of use.

You need to generate a sentence that resembles clauses from terms of use contracts.
The examples are NOT allowed to be a arbitration clause. 

NEW EXAMPLE:



In [ ]:

positive_prompts_few_shot = generate_positive_prompts_few_shot(clause_type, definitions, train, num_examples)
negative_prompts_few_shot = generate_negative_prompts_few_shot(clause_type, definitions, train, num_examples)
print(negative_prompts_few_shot[0])

You are a helpful AI that creates synthetic data by generating legal clauses for terms of use.

You need to generate a sentence that resembles example clauses provided below. You will have to rephrase them using different words.
The examples are NOT allowed to be a arbitration clause.  

EXAMPLES:          
Below are three examples clause. One line per example.

OkCupid does not take part in dispute settlement procedures in front of a consumer arbitration entity for users residing in the EU, EEA, UK, or Switzerland.
Any Disputes that are not subject to the Arbitration Terms or that are severed from any arbitration may only be litigated in the federal or state courts of San Mateo County, California; and the parties consent to personal and exclusive jurisdiction in these courts, except as otherwise provided by the GDPR.
For any action at law or in equity relating to the arbitration provision of these Terms, the Excluded Disputes or if you opt out of the agreement to arbitrate, you agree 

In [155]:

positive_prompts_contrastive_few_shot = generate_positive_prompts_contrastive_few_shot(clause_type, definitions, train, num_examples)
negative_prompts_contrastive_few_shot = generate_negative_prompts_contrastive_few_shot(clause_type, definitions, train, num_examples)
print(negative_prompts_contrastive_few_shot[0])

You are a helpful AI that creates synthetic data by generating legal clauses for terms of use.

We provide you with the following information:
- EXAMPLES OF arbitration CLAUSES: We give three examples of arbitration clauses
- CONTRASTIVE EXAMPLES: We give three contrastive examples, which are text fragments that might resemble arbitration clauses BUT DO NOT have the same legal meaning or function. 

You need to generate a sentence that is similar to these constrastive examples but uses different words. 
The examples are NOT allowed to be a arbitration clause.

EXAMPLES OF arbitration CLAUSES:          
Below are three examples of a arbitration clause. One example per line.

  THIS AGREEMENT REQUIRES FINAL AND BINDING ARBITRATION TO RESOLVE ANY DISPUTE OR CLAIM ARISING OUT OF OR RELATING IN ANY WAY TO THIS AGREEMENT, OR YOUR ACCESS TO OR USE OF THE RUMBLE SERVICES, INCLUDING THE VALIDITY, APPLICABILITY OR INTERPRETATION OF THIS AGREEMENT, AND YOU AGREE THAT ANY SUCH CLAIM WILL BE RESOLV

In [156]:
client = OpenAI()

dfs = []

for task, prompts in [('positive_prompts_zero_shot', positive_prompts_zero_shot),
                      ('negative_prompts_zero_shot', negative_prompts_zeo_shot),
                      ('positive_prompts_few_shot', positive_prompts_few_shot),
                      ('negative_prompts_few_shot', negative_prompts_few_shot),
                      ('positive_prompts_contrastive_few_shot', positive_prompts_contrastive_few_shot),
                      ('negative_prompts_contrastive_few_shot', negative_prompts_contrastive_few_shot)]:
    print(task)
    df_temp =  pd.DataFrame(
                        [(p,generate_response(p,client,model='gpt-4o',max_tokens=100, temperature=0.5)) for p in tqdm(prompts)],
                        columns=['prompt','text']
                        )
    
    df_temp['task'] = task
    
    if task.startswith('positive'):
        df_temp['labels'] = 1
    else:
        df_temp['labels'] = 0

    dfs.append(df_temp)


negative_prompts_zero_shot


  0%|          | 0/100 [00:00<?, ?it/s]

negative_prompts_few_shot


  0%|          | 0/100 [00:00<?, ?it/s]

negative_prompts_contrastive_few_shot


  0%|          | 0/100 [00:00<?, ?it/s]

In [157]:
pd.concat(dfs, axis=0, ignore_index=True).to_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4.csv')
#pd.concat(dfs, axis=0, ignore_index=True).to_csv(f'annotations/synthetic/{clause_type}_synthetic_negative_gpt4.csv')

In [162]:
# # temporary code to merge negative and positive examples
# syntethic_data_all = pd.read_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4.csv', index_col=0)
# print(syntethic_data_all.shape)
# snythetic_dat_neg = pd.read_csv(f'annotations/synthetic/{clause_type}_synthetic_negative_gpt4.csv', index_col=0)
# print(snythetic_dat_neg.shape)
# syntethic_data_all = pd.concat([syntethic_data_all[syntethic_data_all.task.str.startswith('positive')], snythetic_dat_neg], axis=0, ignore_index=True)
# print(syntethic_data_all.shape, syntethic_data_all.task.value_counts())
# syntethic_data_all.to_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4_updated.csv')

(600, 4)
(300, 4)
(600, 4) task
positive_prompts_zero_shot               100
positive_prompts_few_shot                100
positive_prompts_contrastive_few_shot    100
negative_prompts_zero_shot               100
negative_prompts_few_shot                100
negative_prompts_contrastive_few_shot    100
Name: count, dtype: int64


### Second iteration

In [219]:
syntethic_data = pd.read_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4.csv', index_col=0)

In [220]:
syntethic_data.task.unique()

array(['positive_prompts_zero_shot', 'negative_prompts_zero_shot',
       'positive_prompts_few_shot', 'negative_prompts_few_shot',
       'positive_prompts_contrastive_few_shot',
       'negative_prompts_contrastive_few_shot'], dtype=object)

In [221]:
from tools.prompt_templates import generate_positive_prompts_few_shot_syn, generate_negative_prompts_few_shot_syn
positive_prompts_few_shot_syn = generate_positive_prompts_few_shot_syn(clause_type, definitions, train,syntethic_data, num_examples)
negative_prompts_few_shot_syn = generate_negative_prompts_few_shot_syn(clause_type, definitions, train, syntethic_data,num_examples)
print(positive_prompts_few_shot_syn[0], len(positive_prompts_few_shot_syn))

You are a helpful AI that creates synthetic data by generating a new example a class waiver clause.

We first give a definition of a class waiver clause followed by five synthetic examples. 
You need to generate a new example of a class waiver clause that has the legal function and meaning but uses different words than these synthetic examples. 
               
DEFINITON: 
The defintion of a class waiver clause is: A class action waiver is a provision found in some contracts which prohibits a party from filing a class action legal proceeding against the other party, or both parties waiving the right to file class actions against each other. Most class action waiver clauses include this wording or a variation of it: You and we agree that any dispute filed against each other must be on an individual basis and not as a class or collective action.

SYNTHETIC EXAMPLES:
Below are five synthetic examples of a class waiver clause. One line per example.
 
Make sure your new example is uses diff

In [222]:
print(negative_prompts_few_shot_syn[0])

You are a helpful AI that creates synthetic data by generating a contrastive example of class waiver clauses, which is a short text that looks like a class waiver clause but legally has a different function and meaning.

We first give a definition of a class waiver clause followed by five synthetic contrastive examples, which are text fragments that resemble class waiver clauses BUT DO NOT have the same legal meaning or function. 
You need to generate a sentence that is similar to these constrastive examples but using different words.
The examples are NOT allowed to be a class waiver clause.  
            
DEFINITON: 
The defintion of of class waiver clause is: A class action waiver is a provision found in some contracts which prohibits a party from filing a class action legal proceeding against the other party, or both parties waiving the right to file class actions against each other. Most class action waiver clauses include this wording or a variation of it: You and we agree that an

In [223]:
client = OpenAI()

dfs = []

for task, prompts in [('positive_prompts_few_shot_syn', positive_prompts_few_shot_syn),
                      ('negative_prompts_few_shot_syn', negative_prompts_few_shot_syn),
                      ]:
    print(task)
    df_temp =  pd.DataFrame(
                        [(p,generate_response(p,client,model='gpt-4o',max_tokens=100, temperature=0.5)) for p in tqdm(prompts)],
                        columns=['prompt','text']
                        )
    
    df_temp['task'] = task
    
    if task.startswith('positive'):
        df_temp['labels'] = 1
    else:
        df_temp['labels'] = 0

    dfs.append(df_temp)


positive_prompts_few_shot_syn


  0%|          | 0/100 [00:00<?, ?it/s]

negative_prompts_few_shot_syn


  0%|          | 0/100 [00:00<?, ?it/s]

In [224]:
pd.concat(dfs, axis=0, ignore_index=True).to_csv(f'annotations/synthetic/{clause_type}_synthetic_iteration2_gpt4.csv')


# Fin.

# Train and evaluate classifier

## Add synthetic data and train the model

In [164]:
import torch

def train_model(clause_type,train,test,prompt_type,checkpoint= "distilbert-base-uncased", num_epochs = 10):
    def tokenize_function(example):
        return tokenizer(example["text"], truncation=True)
    
    scores_all = []

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "mps" if torch.backends.mps.is_available() else "cpu"
        )

    torch.manual_seed(1984)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
    model.to(device)
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)

    # Combine into a DatasetDict
    dataset = DatasetDict({
        'train':  Dataset.from_pandas(train),
        'test': Dataset.from_pandas(test[['text','labels']].reset_index(drop=True)),
    })

    tokenized_datasets = dataset.map(tokenize_function, batched=True)
    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    tokenized_datasets = tokenized_datasets.remove_columns(["text"])
    tokenized_datasets.set_format("torch")

    train_dataloader = DataLoader(
        tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
            )
    eval_dataloader = DataLoader(
        tokenized_datasets["test"], batch_size=8, collate_fn=data_collator
        )
    
    optimizer = AdamW(model.parameters(), lr=1e-5, eps=1e-6, weight_decay=0.2)

    
    num_training_steps = num_epochs * len(train_dataloader)

    lr_scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=2,
        num_training_steps=num_training_steps,
    )

    progress_bar = tqdm(range(num_training_steps))
    metric = evaluate.load("glue", "mrpc")

    model.train()
    best_score = .0
    for epoch in range(num_epochs):
        for batch in train_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            loss.backward()

            optimizer.step()
            lr_scheduler.step()
            optimizer.zero_grad()
            progress_bar.update(1)


        model.eval()
        for batch in eval_dataloader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.no_grad():
                outputs = model(**batch)

            logits = outputs.logits
            predictions = torch.argmax(logits, dim=-1)
            metric.add_batch(predictions=predictions, references=batch["labels"])

        scores = metric.compute()
        scores_all.append(scores)
        print(f"Epoch {epoch}:", scores)

        if scores["f1"] > best_score:
            print("Saving model")
            best_score = scores["f1"] 
            model.save_pretrained(f"./models/{clause_type}_model")
            tokenizer.save_pretrained(f"./models/{clause_type}_{prompt_type}_model")
    return scores_all, best_score
   


## Experiment per clause

In [171]:
pattern_dict = {'arbitration': re.compile(r'\b(arbitr.*?)\b',re.I), 'opt-out': re.compile(r'\b(opt[\-\s]out)\b',re.I) ,
                'class waiver': re.compile(r'\b(class.*?|individu.*?|waiv.*?)\b',re.I)}
MASK = False

# optional add randomly samples examples as negative examples
df_sample_train = metadata.sample(n=200, random_state=0).reset_index(drop=True)
train_sents = [s for s in df_sample_train.sentence_original.to_list() if s not in train.text.to_list()]
df_sample_train = pd.DataFrame(train_sents, columns=['text'])
df_sample_train['labels'] = 0
train_data = pd.concat([train[['text','labels']], df_sample_train[['text','labels']] ], axis=0, ignore_index=True).reset_index(drop=True)


In [172]:


df_sample_test = metadata.sample(n=50, random_state=2).reset_index(drop=True)
test_sents = [s for s in df_sample_test.sentence_original.to_list() if s not in train_data.text.to_list()]
df_sample_test = pd.DataFrame(train_sents, columns=['text'])
df_sample_test['labels'] = 0
test_data = pd.concat([test[['text','labels']], df_sample_test[['text','labels']] ], axis=0, ignore_index=True).reset_index(drop=True)


syntethic_data_all = pd.read_csv(f'annotations/synthetic/{clause_type}_synthetic_gpt4_updated.csv', index_col=0)
syntethic_data_all.task.replace({'negative_prompts_zeo_shot': 'negative_prompts_zero_shot'}, inplace=True)
syntethic_data_all['task'] = syntethic_data_all.task.apply(lambda x: '_'.join(x.split('_')[2:]))

if MASK:
    train_data['text'] = train_data['text'].apply(lambda x: pattern_dict[clause_type].sub( '', x))
    test_data['text'] = test_data['text'].apply(lambda x: pattern_dict[clause_type].sub( '', x))
    syntethic_data_all['text'] = syntethic_data_all['text'].apply(lambda x: pattern_dict[clause_type].sub( '', x))


In [173]:
syntethic_data_all.task.unique()

array(['zero_shot', 'few_shot', 'contrastive_few_shot'], dtype=object)

In [174]:
train_data.labels.value_counts(), test_data.labels.value_counts(), syntethic_data_all.labels.value_counts()

(labels
 0    205
 1     19
 Name: count, dtype: int64,
 labels
 0    210
 1     64
 Name: count, dtype: int64,
 labels
 1    300
 0    300
 Name: count, dtype: int64)

In [175]:

results = {}
results['train_set_only'] = train_model(clause_type,train_data,test_data,'train_set_only')

for prompt_type in syntethic_data_all.task.unique():
    # add synthetic data
    #if prompt_type.startswith('positive'):
        syntethic_data = syntethic_data_all[syntethic_data_all.task==prompt_type]
        train_data_syn = pd.concat([train_data, syntethic_data[['text','labels']]], axis=0, ignore_index=True).reset_index(drop=True)
        results[prompt_type] = train_model(clause_type,train_data_syn,test_data,prompt_type)

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/224 [00:00<?, ? examples/s]

Map:   0%|          | 0/274 [00:00<?, ? examples/s]

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/280 [00:00<?, ?it/s]

...

Map:   0%|          | 0/424 [00:00<?, ? examples/s]

Map:   0%|          | 0/274 [00:00<?, ? examples/s]

/Users/kasparbeelen/anaconda3/envs/tou/lib/python3.10/site-packages/transformers/optimization.py:429: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(


  0%|          | 0/530 [00:00<?, ?it/s]

Epoch 0: {'accuracy': 0.7846715328467153, 'f1': 0.25316455696202533}
Saving model
Epoch 1: {'accuracy': 0.8686131386861314, 'f1': 0.625}
Saving model
Epoch 2: {'accuracy': 0.9051094890510949, 'f1': 0.7636363636363637}
Saving model
Epoch 3: {'accuracy': 0.9233576642335767, 'f1': 0.8173913043478261}
Saving model
Epoch 4: {'accuracy': 0.8941605839416058, 'f1': 0.7238095238095238}
Epoch 5: {'accuracy': 0.9014598540145985, 'f1': 0.7476635514018691}
Epoch 6: {'accuracy': 0.9014598540145985, 'f1': 0.7476635514018691}
Epoch 7: {'accuracy': 0.9051094890510949, 'f1': 0.7592592592592593}
Epoch 8: {'accuracy': 0.9051094890510949, 'f1': 0.7592592592592593}
Epoch 9: {'accuracy': 0.9051094890510949, 'f1': 0.7592592592592593}


In [97]:
res = []
for k,v in results.items():
    res.append([k,v[1]])
pd.DataFrame(res, columns=['prompt','f1'])

,prompt,f1
0,train_set_only,0.836735
1,zero_shot,0.833333
2,few_shot,0.822222
3,contrastive_few_shot,0.817204


## Experiments combined

# Apply Classifier

In [ ]:
from torch.nn.functional import softmax
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from transformers import AutoTokenizer, DataCollatorWithPadding

In [ ]:
processed_data_dir = Path('processed_data/multigenre')
file_name = processed_data_dir / 'metadata.tsv'
metadata = pd.read_csv(file_name,sep='\t')
metadata.fillna('', inplace=True)
print(len(metadata))

149261


In [ ]:
clause_type = 'anti-scraping'
model = AutoModelForSequenceClassification.from_pretrained(f'models/{clause_type}_model')
tokenizer = AutoTokenizer.from_pretrained(f'models/{clause_type}_model')

In [ ]:
metadata.head()

,file_name,platform,genre,tag,sentence,logits_arbitration,prob_1_arbitration,arbitration
0,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,privacy policy effective date : [mask] [mask] ...,"[[tensor(0.9787), tensor(0.0213)]]",0.021337,0.0
1,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data controller is [mask] [mask] and can b...,"[[tensor(0.9763), tensor(0.0237)]]",0.023692,0.0
2,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data protection officer is : [mask] [mask]...,"[[tensor(0.9772), tensor(0.0228)]]",0.022761,0.0
3,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,[mask] ec4 m 7ef sciplaydpo@scientificgames.co...,"[[tensor(0.9687), tensor(0.0313)]]",0.031294,0.0
4,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,you may also use the link found at the end of ...,"[[tensor(0.9769), tensor(0.0231)]]",0.023066,0.0


In [ ]:
metadata.iloc[137375:137380]

,file_name,platform,genre,tag,sentence,logits_arbitration,prob_1_arbitration,arbitration
137375,tumblr_tos_social,tumblr,social,ToS,open source disclosures \n you can find disclo...,"[[tensor(0.9766), tensor(0.0234)]]",0.023355,0.0
137376,tumblr_tos_social,tumblr,social,ToS,,"[[tensor(0.7748), tensor(0.2252)]]",0.225161,0.0
137377,huggingface_tou_ai,huggingface,ai,ToS,thanks for using [mask] [mask] and being part ...,"[[tensor(0.9522), tensor(0.0478)]]",0.047831,0.0
137378,huggingface_tou_ai,huggingface,ai,ToS,we drafted the following [mask] [mask] [mask] ...,"[[tensor(0.9751), tensor(0.0249)]]",0.024918,0.0
137379,huggingface_tou_ai,huggingface,ai,ToS,we are very much open to feedback - contact us...,"[[tensor(0.9687), tensor(0.0313)]]",0.031342,0.0


In [ ]:
tqdm.pandas()
metadata[f'logits_{clause_type}'] = metadata.progress_apply(lambda x: 
                    softmax(model(**tokenizer(x.sentence, return_tensors='pt', truncation=True)).logits.detach(), dim=1), 
                          axis=1)

  0%|          | 0/149261 [00:00<?, ?it/s]

100%|██████████| 149261/149261 [1:12:32<00:00, 34.29it/s] 


In [ ]:
metadata[f'prob_1_{clause_type}'] = metadata[f'logits_{clause_type}'].apply(lambda x: x[0][1].item())
metadata[clause_type] = .0
metadata.loc[metadata[f'prob_1_{clause_type}'] > .5, clause_type] = 1

In [ ]:
metadata.drop(columns=['logits_arbitration', 'logits_anti-scraping'], inplace=True)

In [ ]:
metadata.to_csv(processed_data_dir /f'{file_name.stem}_annotated.tsv', sep='\t', index=False)

In [ ]:
metadata.head()

,file_name,platform,genre,tag,sentence,prob_1_arbitration,arbitration,prob_1_anti-scraping,anti-scraping
0,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,privacy policy effective date : [mask] [mask] ...,0.021337,0.0,0.014945,0.0
1,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data controller is [mask] [mask] and can b...,0.023692,0.0,0.021128,0.0
2,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,the data protection officer is : [mask] [mask]...,0.022761,0.0,0.016152,0.0
3,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,[mask] ec4 m 7ef sciplaydpo@scientificgames.co...,0.031294,0.0,0.028223,0.0
4,JackpotPartyCasinoSlotsPP_gaming,JackpotPartyCasinoSlotsPP,gaming,PP,you may also use the link found at the end of ...,0.023066,0.0,0.019758,0.0


In [ ]:
#metadata.sort_values('prob_1', ascending=False).head(10)

In [ ]:
df_deduplicated = metadata.drop_duplicates(subset=['sentence'])
df_deduplicated['annotated'] = df_deduplicated.sentence.isin(df_annotations.text)
int_labels = [((0.95,1.0),'confident_positive'),( (0.80,.95), 'sure_positive'),((0.60,.80), 'leaning_positive'),
                   ((0.50,.60), 'borderline_positive'),((0.40,.50), 'borderline_negative'),
                   ((0.20,.40), 'leaning_negative'),((0.05,.20), 'sure_negative'),((0.0,.05), 'confident_negative')]
for interval, label in int_labels:


    df_deduplicated.loc[df_deduplicated.prob_1.between(*interval),'category']  = label



In [ ]:
pd.concat([df_deduplicated[df_deduplicated.category == label].sample(10)
    for _ , label in int_labels], axis=0)[['sentence','category']].to_csv(f'annotations/inference/{clause_type}_automatic_annotations_by_category.csv')


In [ ]:
metadata[metadata.prob_1 > .5].to_csv(f'annotations/inference/{clause_type}_inference.csv')